In [ ]:
import os
import json
import pandas as pd
import csv
from dotenv import load_dotenv
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from openai import OpenAI

# Load environment variables
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

class ScriptAnalyzer:
    def __init__(self, openai_api_key: str = None):
        """Initialize the ScriptAnalyzer and related components."""
        self.api_key = openai_api_key or os.getenv("OPENAI_API_KEY")
        self.client = OpenAI(api_key=self.api_key)
        
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1500,
            chunk_overlap=200,
            separators=["\n\n", "\n", ". ", " ", ""]
        )
        
        self.embeddings = OpenAIEmbeddings(
            model="text-embedding-ada-002",
            openai_api_key=self.api_key
        )

    def load_script(self, script_path: str) -> str:
        """Load script text from a file."""
        with open(script_path, 'r', encoding='utf-8') as file:
            return file.read()

    def chunk_text(self, text: str) -> list:
        """Split text into chunks for processing."""
        return self.text_splitter.create_documents([text])

    def create_vector_store(self, documents: list):
        """Create a vector store from document chunks."""
        return FAISS.from_documents(documents, self.embeddings)

    def get_relevant_context(self, vector_store, query: str, k: int = 5) -> str:
        """Retrieve relevant context based on a query."""
        docs = vector_store.similarity_search(query, k=k)
        return "\n\n".join([doc.page_content for doc in docs])
    
    def load_character_details(self, json_path):
        """Load character details from JSON file."""
        with open(json_path, 'r', encoding='utf-8') as file:
            return json.load(file)
    
    def write_relationships_csv(self, filepath, relationships):
        """Writes relationship triples to CSV."""
        with open(filepath, "w", newline='', encoding='utf-8') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(["source", "relationship", "target"])  # Header
            writer.writerows(relationships)
    
    def extract_relationships_with_rag(self, script_path, character_details_json, output_csv):
        """Use RAG to analyze the script and extract relationships based on character details JSON."""
        # Load and chunk the script
        script_text = self.load_script(script_path)
        script_chunks = self.chunk_text(script_text)
        
        # Create vector store
        vector_store = self.create_vector_store(script_chunks)
        
        # Load character data from JSON
        characters = self.load_character_details(character_details_json)
        
        # Extract relationships for each character pair
        all_relationships = []
        
        # Format character information for the prompt
        char_list = ""
        for i, char in enumerate(characters):
            name = char["character"]
            normalized_name = char["normalized_name"]
            mentions = char["mentions"]
            
            # Extract key personality traits if available
            personality_traits = []
            if "analysis" in char and "character" in char["analysis"] and "about" in char["analysis"]["character"] and "personality" in char["analysis"]["character"]["about"]:
                personality = char["analysis"]["character"]["about"]["personality"]
                if "personalityTraits" in personality:
                    personality_traits = personality["personalityTraits"]
            
            # Format character information
            char_info = f"{i+1}. {name} (normalized: {normalized_name}, mentions: {mentions})"
            if personality_traits:
                char_info += f", traits: {', '.join(personality_traits)}"
            char_list += char_info + "\n"
        
        # Get relationship context using RAG
        relationship_query = "Find sections where characters interact or relationships are established"
        relevant_script_sections = self.get_relevant_context(vector_store, relationship_query, k=10)
        
        # Create the prompt with RAG-retrieved context
        prompt = (
            f"You are an expert in analyzing film scripts and extracting relationships between entities. "
            f"I will provide you with a film script, a list of major characters, and your task is to extract relationships between characters.\n"
            f"Here are all the characters to focus on:\n{char_list}\n\n"
            f"Here are the most relevant sections of the script showing character interactions:\n\n"
            f"{relevant_script_sections}\n\n"
            f"Output the relationships as triples in the format 'Entity1,Relationship,Entity2' (one per line). "
            f"Use the normalized character names in your output. "
            f"Focus on extracting only the 15-20 most important relationships that are clearly supported by the script between all types of characters. "
            f"Do not invent relationships that aren't evident in the text."
        )

        # Call OpenAI API
        print("Sending relationship extraction prompt to OpenAI API...")
        response = self.client.chat.completions.create(
            model="gpt-4",
            messages=[
                {"role": "system", "content": "You are an expert in analyzing film scripts and extracting relationships between characters."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.3,
            max_tokens=1000
        )
        response_text = response.choices[0].message.content

        response_text = response_text.replace("```", "").strip()

        # Parse the relationships
        relationships = []
        for line in response_text.splitlines():
            line = line.strip()
            if line and "," in line:
                parts = line.split(",")
                if len(parts) >= 3:
                    source = parts[0].strip()
                    relationship = parts[1].strip()
                    target = parts[2].strip()
                    relationships.append([source, relationship, target])

        # Write relationships to CSV
        self.write_relationships_csv(output_csv, relationships)
        print(f"Extracted {len(relationships)} relationships and saved to {output_csv}")
        
        return relationships, characters


def main():
    SCRIPT_FILE = "film_script2.txt"
    CHARACTER_DETAILS_JSON = "character_details.json"
    OUTPUT_RELATIONSHIPS = "relationships33.csv"

    
    # Initialize ScriptAnalyzer
    analyzer = ScriptAnalyzer()
    
    # Extract relationships using RAG and OpenAI
    relationships, characters = analyzer.extract_relationships_with_rag(
        SCRIPT_FILE, 
        CHARACTER_DETAILS_JSON, 
        OUTPUT_RELATIONSHIPS
    )
    
  
    
    print(f"Process complete. {len(relationships)} relationships extracted and visualized.")

if __name__ == "__main__":
    main()

Sending relationship extraction prompt to OpenAI API...
Extracted 30 relationships and saved to relationships33.csv
Enhanced knowledge graph visualization created at character_relationships33.html
Process complete. 30 relationships extracted and visualized.


In [7]:
import pandas as pd
import networkx as nx
from pyvis.network import Network

# Load CSV file
df = pd.read_csv("relationships3.csv")

# Create a directed graph
G = nx.DiGraph()

# Add edges with labels
for _, row in df.iterrows():
    G.add_edge(row["source"], row["target"], label=row["relationship"])

# Create a Pyvis network with inline CDN resources
net = Network(notebook=True, directed=True, cdn_resources='remote')

# Add nodes and edges with labels
for node in G.nodes():
    net.add_node(node, label=node)

for edge in G.edges(data=True):
    net.add_edge(edge[0], edge[1], title=edge[2]["label"], label=edge[2]["label"])  # Add edge label

# Save the network as HTML with UTF-8 encoding
net.show("knowledge_graph2.html")

# Reopen the file with UTF-8 encoding and write it back with UTF-8
with open("knowledge_graph2.html", "r", encoding="utf-8") as file:
    html_content = file.read()

with open("knowledge_graph2.html", "w", encoding="utf-8") as file:
    file.write(html_content)


knowledge_graph2.html


In [ ]:
# file:///c:/Users/Pc/Downloads/plotdotai-chat/step4_knowledge_graphs/knowledge_graph1.html
